# Rain Prediction with an ANN (fixed for latest TensorFlow / Keras 3 / pandas 3)

Place `weatherAUS.csv` and `rain_utils.py` in the same folder as this notebook. Run all cells; the last cell saves the model and preprocessor used by the Streamlit app (`app.py`).

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import tensorflow as tf
from tensorflow import keras
from keras import layers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from rain_utils import (RainPreprocessor, CAT_COLS, NUM_COLS, FEATURE_COLS,
                        add_date_features, outlier_mask)

np.random.seed(0)
tf.random.set_seed(0)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| pandas", pd.__version__)

## 2. Load data

In [ ]:
DATA_PATH = "weatherAUS.csv"   # change if your file lives elsewhere
data = pd.read_csv(DATA_PATH)

# Rows without a target cannot be used for supervised training.
data = data.dropna(subset=["RainTomorrow"]).reset_index(drop=True)
print(data.shape)
data.head()

In [ ]:
data.info()

## 3. Visualisation

In [ ]:
cols = ["#C2C4E2", "#EED4E5"]
sns.countplot(x="RainTomorrow", data=data, hue="RainTomorrow", palette=cols, legend=False)
plt.title("Target balance")
plt.show()

In [ ]:
corrmat = data.corr(numeric_only=True)          # numeric_only is required on modern pandas
cmap = sns.diverging_palette(260, -10, s=50, l=75, n=6, as_cmap=True)
plt.figure(figsize=(16, 16))
sns.heatmap(corrmat, cmap=cmap, annot=True, fmt=".2f", square=True)
plt.show()

In [ ]:
tmp = add_date_features(data.assign(Date=pd.to_datetime(data["Date"])))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.scatterplot(x="month_sin", y="month_cos", data=tmp, color="#C2C4E2", ax=ax[0]).set_title("Cyclic month")
sns.scatterplot(x="day_sin", y="day_cos", data=tmp, color="#C2C4E2", ax=ax[1]).set_title("Cyclic day")
plt.show()

plt.figure(figsize=(12, 6))
sns.lineplot(x=tmp["year"], y=tmp["Rainfall"], color="#C2C4E2")
plt.title("Rainfall over years"); plt.show()

## 4. Preprocessing
Missing values are filled (mode for categories, median for numbers), categories are label-encoded, month/day are encoded cyclically and everything is standardised. All of this lives in `RainPreprocessor` so the app can reuse it.

In [ ]:
pre = RainPreprocessor().fit(data)
features = pre.transform(data)
target = (data["RainTomorrow"] == "Yes").astype("float32")   # No=0, Yes=1

# Remove outliers (limits from the original notebook, in standardised units)
keep = outlier_mask(features)
X = features[keep].astype("float32")
y = target[keep].astype("float32")
print("Rows kept:", X.shape[0], "of", len(features), "| features:", X.shape[1])

In [ ]:
plt.figure(figsize=(18, 7))
sns.boxenplot(data=X, color="#C2C4E2")
plt.xticks(rotation=90); plt.show()

## 5. Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X.to_numpy(), y.to_numpy(), test_size=0.2, random_state=42, stratify=y
)

early_stopping = callbacks.EarlyStopping(
    monitor="val_loss", min_delta=0.001, patience=20, restore_best_weights=True
)

model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation="relu", kernel_initializer="uniform"),
    layers.Dense(32, activation="relu", kernel_initializer="uniform"),
    layers.Dense(16, activation="relu", kernel_initializer="uniform"),
    layers.Dropout(0.25),
    layers.Dense(8, activation="relu", kernel_initializer="uniform"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid", kernel_initializer="uniform"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.00009),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=150,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=2,
)

In [ ]:
history_df = pd.DataFrame(history.history)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history_df["loss"], color="#BDE2E2", label="Training loss")
ax[0].plot(history_df["val_loss"], color="#C2C4E2", label="Validation loss")
ax[0].set(title="Loss", xlabel="Epochs", ylabel="Loss"); ax[0].legend()
ax[1].plot(history_df["accuracy"], color="#BDE2E2", label="Training accuracy")
ax[1].plot(history_df["val_accuracy"], color="#C2C4E2", label="Validation accuracy")
ax[1].set(title="Accuracy", xlabel="Epochs", ylabel="Accuracy"); ax[1].legend()
plt.show()

## 6. Evaluation

In [ ]:
proba = model.predict(X_test, verbose=0).ravel()
pred = (proba > 0.5).astype(int)
print("Test accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, target_names=["No rain", "Rain"]))
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=["No rain", "Rain"], yticklabels=["No rain", "Rain"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()

## 7. Save for the Streamlit app

In [ ]:
model.save("rain_model.keras")
joblib.dump(pre, "preprocessor.joblib")
print("Saved rain_model.keras and preprocessor.joblib")